# Active Learning Experiment

Этот notebook показывает image active learning поверх уже подготовленного `annotation_artifacts` run.

Важно: в текущем demo-run всего 63 изображения, поэтому для воспроизводимого прогона здесь используются уменьшенные `initial_size`, `batch_size` и `n_iterations`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image as DisplayImage, display

from agents.al_agent import active_learning_op

In [ ]:
run_dir = Path("annotation_artifacts/data_20260320_154130_37a2705c")
config = {
    "artifacts_dir": "al_artifacts",
    "model_path": "yolo11n-cls.pt",
    "initial_size": 30,
    "n_iterations": 2,
    "batch_size": 10,
    "test_size": 9,
    "epochs": 3,
    "imgsz": 224,
    "train_batch": 8,
    "predict_batch": 8,
    "strategies": ["confidence", "random"],
}
config_json = json.dumps(config, ensure_ascii=False)
config

In [ ]:
result = active_learning_op(
    task_description="Image classification: identify bird species from labeled images",
    labeled_data_path=str(run_dir),
    modality="image",
    config_json=config_json,
)
result["summary_path"]

In [ ]:
summary = json.loads(Path(result["summary_path"]).read_text(encoding="utf-8"))
summary

In [ ]:
confidence_history = pd.read_csv(result["strategy_results"]["confidence"]["history_csv"])
random_history = pd.read_csv(result["strategy_results"]["random"]["history_csv"])
display(confidence_history)
display(random_history)
display(DisplayImage(filename=result["learning_curve_path"]))

## Label Studio feedback loop

На каждой итерации confidence-стратегии агент сохраняет:

- `uncertain_manifest.csv`
- `labelstudio_import.json`

Дальше workflow такой:

1. импортировать `labelstudio_import.json` в Label Studio,
2. разметить uncertain batch,
3. сохранить экспорт Label Studio в JSON или CSV,
4. передать путь к нему через `manual_labels_by_iteration` в `config_json`.

Пример структуры:

```python
config["manual_labels_by_iteration"] = {
    "confidence": {
        "1": "/abs/path/to/labelstudio_iteration_1.json",
        "2": "/abs/path/to/labelstudio_iteration_2.json",
    }
}
```